In [5]:
# Going for flow agents
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

In [6]:
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

df = pd.read_csv('cicids2017_cleaned.csv')


In [7]:
df.head

<bound method NDFrame.head of          Destination Port  Flow Duration  Total Fwd Packets  \
0                      22        1266342                 41   
1                      22        1319353                 41   
2                      22            160                  1   
3                      22        1303488                 41   
4                   35396             77                  1   
...                   ...            ...                ...   
2520746                53          32215                  4   
2520747                53            324                  2   
2520748             58030             82                  2   
2520749                53        1048635                  6   
2520750                53          94939                  4   

         Total Length of Fwd Packets  Fwd Packet Length Max  \
0                               2664                    456   
1                               2664                    456   
2                       

In [8]:
df.tail()

,Destination Port,Flow Duration,Total Fwd Packets,Total Length of Fwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,Bwd Packet Length Max,Bwd Packet Length Min,...,Init_Win_bytes_backward,act_data_pkt_fwd,min_seg_size_forward,Active Mean,Active Max,Active Min,Idle Mean,Idle Max,Idle Min,Attack Type
2520746,53,32215,4,112,28,28,28.0,0.00000,76,76,...,-1,3,20,0.0,0,0,0.0,0,0,Normal Traffic
2520747,53,324,2,84,42,42,42.0,0.00000,181,181,...,-1,1,20,0.0,0,0,0.0,0,0,Normal Traffic
2520748,58030,82,2,31,31,0,15.5,21.92031,6,6,...,0,0,32,0.0,0,0,0.0,0,0,Normal Traffic
2520749,53,1048635,6,192,32,32,32.0,0.00000,128,128,...,-1,5,20,0.0,0,0,0.0,0,0,Normal Traffic
2520750,53,94939,4,188,47,47,47.0,0.00000,113,113,...,-1,3,20,0.0,0,0,0.0,0,0,Normal Traffic


In [17]:
df.columns = df.columns.str.strip()

# Use only normal traffic rows for training
normal = df[df['Attack Type'] == 'Normal Traffic']

# Use the columns that map closest to your 7 features
FEATURES = [
    'Flow Duration',
    'Total Fwd Packets',
    'Flow Bytes/s',
    'Flow Packets/s',
    'Flow IAT Std',   # inter-arrival time std — beaconing signal
]

X_normal = normal[FEATURES].replace([float('inf'), -float('inf')], 0).fillna(0)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_normal)

model = IsolationForest(n_estimators=200, contamination=0.05, random_state=42)
model.fit(X_scaled)
print("Isolation Forest trained on", len(X_scaled), "normal flows")

Isolation Forest trained on 2095057 normal flows


In [10]:
df.columns

Index(['Destination Port', 'Flow Duration', 'Total Fwd Packets',
       'Total Length of Fwd Packets', 'Fwd Packet Length Max',
       'Fwd Packet Length Min', 'Fwd Packet Length Mean',
       'Fwd Packet Length Std', 'Bwd Packet Length Max',
       'Bwd Packet Length Min', 'Bwd Packet Length Mean',
       'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s',
       'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min',
       'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max',
       'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std',
       'Bwd IAT Max', 'Bwd IAT Min', 'Fwd Header Length', 'Bwd Header Length',
       'Fwd Packets/s', 'Bwd Packets/s', 'Min Packet Length',
       'Max Packet Length', 'Packet Length Mean', 'Packet Length Std',
       'Packet Length Variance', 'FIN Flag Count', 'PSH Flag Count',
       'ACK Flag Count', 'Average Packet Size', 'Subflow Fwd Bytes',
       'Init_Win_bytes_forward', 'Init_Win_bytes_backward', 'act_data_p

In [18]:
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
import numpy as np

# Same feature preparation
X_test = df[FEATURES].replace(
    [float('inf'), -float('inf')], 0
).fillna(0)


# True labels
y_true = (
    df['Attack Type'] != 'Normal Traffic'
).astype(int)


# Scale using SAME scaler (do not fit again)
X_test_scaled = scaler.transform(X_test)

In [19]:
y_pred_if = model.predict(X_test_scaled)

y_pred = np.where(y_pred_if == -1, 1, 0)

In [20]:
print(
    classification_report(
        y_true,
        y_pred,
        target_names=['Normal','Attack']
    )
)

              precision    recall  f1-score   support

      Normal       0.88      0.95      0.91   2095057
      Attack       0.60      0.37      0.46    425694

    accuracy                           0.85   2520751
   macro avg       0.74      0.66      0.69   2520751
weighted avg       0.83      0.85      0.84   2520751



In [21]:
anomaly_score = -model.decision_function(X_test_scaled)

In [22]:
auc = roc_auc_score(
    y_true,
    anomaly_score
)

print("AUROC:", auc)

AUROC: 0.729251245927116


In [23]:
print(
    confusion_matrix(
        y_true,
        y_pred
    )
)

[[1990308  104749]
 [ 267969  157725]]


In [30]:
# supervised part of the problem
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

# Use all rows — both normal and attack
X = df[FEATURES].replace([float('inf'), -float('inf')], 0).fillna(0)
y = (df['Attack Type'] != 'Normal Traffic').astype(int)   # 1=attack, 0=normal

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

n_normal = (y_train==0).sum()
n_attack = (y_train==1).sum()

model = xgb.XGBClassifier(
    n_estimators=500,
    scale_pos_weight = n_normal/n_attack,
    eval_metric='auc',
    early_stopping_rounds=30,
    random_state=42
)
model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=100)

auc = roc_auc_score(y_val, model.predict_proba(X_val)[:,1])
print(f"Validation AUROC: {auc:.4f}")

[0]	validation_0-auc:0.97568
[100]	validation_0-auc:0.99876
[200]	validation_0-auc:0.99898
[300]	validation_0-auc:0.99906
[400]	validation_0-auc:0.99910
[499]	validation_0-auc:0.99912
Validation AUROC: 0.9991


In [29]:
df.head()

,Destination Port,Flow Duration,Total Fwd Packets,Total Length of Fwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,Bwd Packet Length Max,Bwd Packet Length Min,...,Init_Win_bytes_backward,act_data_pkt_fwd,min_seg_size_forward,Active Mean,Active Max,Active Min,Idle Mean,Idle Max,Idle Min,Attack Type
0,22,1266342,41,2664,456,0,64.975610,109.864573,976,0,...,243,24,32,0.0,0,0,0.0,0,0,Normal Traffic
1,22,1319353,41,2664,456,0,64.975610,109.864573,976,0,...,243,24,32,0.0,0,0,0.0,0,0,Normal Traffic
2,22,160,1,0,0,0,0.000000,0.000000,0,0,...,243,0,32,0.0,0,0,0.0,0,0,Normal Traffic
3,22,1303488,41,2728,456,0,66.536585,110.129945,976,0,...,243,24,32,0.0,0,0,0.0,0,0,Normal Traffic
4,35396,77,1,0,0,0,0.000000,0.000000,0,0,...,290,0,32,0.0,0,0,0.0,0,0,Normal Traffic


In [31]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df.columns = df.columns.str.strip()

# Split normal vs attack
normal  = df[df['Attack Type'] == 'Normal Traffic']
attacks = df[df['Attack Type'] != 'Normal Traffic']

# All numeric features available
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

# Compute separation score for each feature
# Separation = how different are the medians between normal and attack
# Higher = more useful for detection
print("Feature separation scores (higher = more useful):")
print("-" * 55)

scores = []
for col in numeric_cols:
    n_med = normal[col].replace([np.inf,-np.inf], np.nan).median()
    a_med = attacks[col].replace([np.inf,-np.inf], np.nan).median()
    denom = abs(n_med) + abs(a_med) + 1e-9
    sep   = abs(a_med - n_med) / denom
    scores.append((col, sep, n_med, a_med))

scores.sort(key=lambda x: x[1], reverse=True)

for col, sep, n_med, a_med in scores[:20]:
    bar = '█' * min(int(sep * 20), 30)
    print(f"  {col:35s} {sep:6.3f}  {bar}")

Feature separation scores (higher = more useful):
-------------------------------------------------------
  Fwd IAT Std                          1.000  ███████████████████
  Bwd IAT Std                          1.000  ███████████████████
  Bwd Packet Length Std                1.000  ███████████████████
  Init_Win_bytes_backward              1.000  ███████████████████
  Fwd Packet Length Std                1.000  ███████████████████
  Fwd Packet Length Min                1.000  ███████████████████
  Bwd Packet Length Min                1.000  ███████████████████
  Min Packet Length                    1.000  ███████████████████
  Fwd IAT Total                        1.000  ███████████████████
  Fwd IAT Max                          1.000  ███████████████████
  Fwd IAT Mean                         1.000  ███████████████████
  Bwd IAT Total                        1.000  ███████████████████
  Bwd IAT Max                          1.000  ███████████████████
  Flow IAT Std                      